In [81]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [82]:
from dotenv import load_dotenv

load_dotenv()

True

In [83]:
import os
from pathlib import Path

import numpy as np

In [84]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
LAYER = 11
DATA_DIR = os.environ["DATA_DIR"]

In [85]:
from reliable_monitoring.dataset import ActivationConfig

activation_config = ActivationConfig(model_name=MODEL_NAME, layer=LAYER)

# reductions debug

In [86]:
from reliable_monitoring.dataset import load_dataset, sample_from_dataset

dataset = load_dataset(
    # Path(f"{DATA_DIR}/training/prompts_4x/train.jsonl"),
    dataset_path=Path(f"{DATA_DIR}/evals/dev/anthropic_balanced_apr_23.jsonl"),
    activation_config=activation_config,
    auto_compute=True,
    cleanup_after_load=True,
)

Deleting activations/baa11b86_11.pt.zst from local
Deleting input_ids/baa11b86.pt.zst from local
Deleting attention_masks/baa11b86.pt.zst from local


In [87]:
acts = dataset.other_fields["activations"]
attn = dataset.other_fields["attention_mask"]

In [88]:
acts.shape

torch.Size([1028, 1861, 2048])

In [89]:
acts[0]

[1861, 2048], tensor([[ 0.0009, -0.0073,  0.0305,  ..., -0.0051,  0.0253, -0.0013],
        [ 0.0009, -0.0073,  0.0305,  ..., -0.0051,  0.0253, -0.0013],
        [-0.0013,  0.0000,  0.4395,  ..., -0.2139,  0.2295, -0.2188],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
       dtype=torch.float16)

In [97]:
for i in range(20):
    nonzero = acts[i].abs().sum(dim=-1) != 0
    # last nonzero position
    last_nonzero_idx = nonzero.nonzero(as_tuple=True)[0][-1]
    print(last_nonzero_idx)
    print(attn[i].sum())
    print('---')

[], tensor(137)
[], tensor(139)
---
[], tensor(176)
[], tensor(178)
---
[], tensor(198)
[], tensor(201)
---
[], tensor(115)
[], tensor(115)
---
[], tensor(77)
[], tensor(76)
---
[], tensor(308)
[], tensor(310)
---
[], tensor(231)
[], tensor(238)
---
[], tensor(144)
[], tensor(148)
---
[], tensor(92)
[], tensor(94)
---
[], tensor(115)
[], tensor(112)
---
[], tensor(157)
[], tensor(159)
---
[], tensor(176)
[], tensor(182)
---
[], tensor(246)
[], tensor(252)
---
[], tensor(274)
[], tensor(280)
---
[], tensor(308)
[], tensor(306)
---
[], tensor(409)
[], tensor(390)
---
[], tensor(198)
[], tensor(199)
---
[], tensor(1237)
[], tensor(883)
---
[], tensor(1237)
[], tensor(722)
---
[], tensor(198)
[], tensor(197)
---


[], tensor(201)

In [47]:
acts.shape

torch.Size([1028, 1861, 2048])

In [48]:
acts[0, -1]

[2048], tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16)

In [49]:
attn.shape

torch.Size([1028, 1861])

In [50]:
print(attn[0])

[1861], tensor([1, 1, 1,  ..., 0, 0, 0], device='cuda:0')


In [51]:
from reliable_monitoring.reductions import reduce_first, reduce_last

In [55]:
last[0].sum()

[], tensor(0., dtype=torch.float16)

In [79]:
acts[:, 50]

[1028, 2048], tensor([[-0.0771,  0.3379,  0.0586,  ...,  0.0391,  0.1709, -0.4434],
        [ 0.1934,  0.2461,  0.4199,  ...,  0.3730, -0.0374, -0.1143],
        [-0.6055, -0.0889,  0.2129,  ..., -0.0732, -0.0703, -0.0874],
        ...,
        [-0.2422, -0.3320, -0.2021,  ...,  0.0564,  0.3320,  0.4375],
        [ 0.4473,  0.0060,  0.4141,  ..., -0.3984, -0.2773,  0.0903],
        [ 0.2988, -0.4297, -0.3613,  ..., -0.2295,  0.1138, -0.1768]],
       dtype=torch.float16)

In [80]:
# What's the tokenizer padding_side setting?                                                                                                             
from transformers import AutoTokenizer                                                                                                                   
tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")                                                                                  
print("padding_side:", tok.padding_side)                                                                                                                 
                                                                                                                                                        
# And verify sample 17's mask around position 883                                                                                                        
print("mask around 883:", attn[17, 880:890])                                                                                                             
print("acts around 883:", acts[17, 880:890].abs().sum(dim=1))

padding_side: right
mask around 883: [10], tensor([1, 1, 1, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')
acts around 883: [10], tensor([753.5000, 769.5000, 747.5000, 772.0000, 758.0000, 732.0000, 732.0000,
        734.0000, 737.0000, 738.5000], dtype=torch.float16)


In [ ]:
nonzero_positions = (acts[17].abs().sum(dim=1) > 0).nonzero().squeeze()                                                                                  
print("Mask sum:", attn[17].sum().item())                                                                                                                
print("Last non-zero pos:", nonzero_positions[-1].item())

Mask sum: 883
Last non-zero pos: 1237


In [54]:
last = reduce_last(acts, attn)
for i in range(20):
    print(last[i])
    

[2048], tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16)
[2048], tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16)
[2048], tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16)
[2048], tensor([-0.5508,  0.2139,  0.4609,  ..., -0.1167,  0.7695, -0.5586],
       dtype=torch.float16)
[2048], tensor([ 0.2949, -0.1318, -0.0713,  ...,  0.4355,  0.6289, -0.4141],
       dtype=torch.float16)
[2048], tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16)
[2048], tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16)
[2048], tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16)
[2048], tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16)
[2048], tensor([-0.1260,  0.3516, -0.0942,  ...,  0.5469,  0.5430, -0.1543],
       dtype=torch.float16)
[2048], tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16)
[2048], tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16)
[2048], tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16)
[2048

In [59]:
attn.sum(dim=1)[:20]

[20], tensor([139, 178, 201, 115,  76, 310, 238, 148,  94, 112, 159, 182, 252, 280,
        306, 390, 199, 883, 722, 197], device='cuda:0')

In [ ]:
print("First 10:", attn[0, :10])                                                                                                                         
print("Last 10:", attn[0, -10:])

First 10: [10], tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1], device='cuda:0')
Last 10: [10], tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')


In [65]:
acts[0, 137]

[2048], tensor([0.1152, 0.1797, 0.1641,  ..., 0.0752, 0.0229, 0.0179],
       dtype=torch.float16)

In [66]:
print("acts device:", acts.device)                                                                                                                       
print("attn device:", attn.device)                                                                                                                       
print("Direct access acts[0, 137]:", acts[0, 137, :10])  # First 10 values at position 138 

acts device: cpu
attn device: cuda:0
Direct access acts[0, 137]: [10], tensor([ 0.1152,  0.1797,  0.1641, -0.0693, -0.2217,  0.0898,  0.4492, -0.1504,
         0.6484, -0.0277], dtype=torch.float16)


In [67]:
# Find the actual last non-zero position for sample 0                                                                                                    
nonzero_positions = (acts[0].abs().sum(dim=1) > 0).nonzero().squeeze()                                                                                   
print("First non-zero pos:", nonzero_positions[0].item())                                                                                                
print("Last non-zero pos:", nonzero_positions[-1].item())                                                                                                
print("Number of non-zero positions:", len(nonzero_positions)) 

First non-zero pos: 0
Last non-zero pos: 137
Number of non-zero positions: 138


In [68]:
nonzero_positions = (acts[17].abs().sum(dim=1) > 0).nonzero().squeeze()                                                                                  
print("Mask sum:", attn[17].sum().item())                                                                                                                
print("Last non-zero pos:", nonzero_positions[-1].item())

Mask sum: 883
Last non-zero pos: 1237


In [57]:
first = reduce_first(acts, attn)
for i in range(20):
    print(first[i])
    

[2048], tensor([ 0.0014, -0.1187,  0.0884,  ..., -0.5898,  0.4082,  0.5664],
       dtype=torch.float16)
[2048], tensor([-0.3164, -0.3750, -0.6367,  ..., -0.2236,  0.3105,  0.3359],
       dtype=torch.float16)
[2048], tensor([-0.4258,  0.3789,  0.8359,  ...,  0.3027, -0.0309,  0.0583],
       dtype=torch.float16)
[2048], tensor([-0.3535, -0.1250, -0.1445,  ...,  0.4844,  0.2734,  0.2988],
       dtype=torch.float16)
[2048], tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16)
[2048], tensor([ 0.2441, -0.5312,  0.4043,  ..., -0.4668,  0.1904, -0.3418],
       dtype=torch.float16)
[2048], tensor([-0.3184, -0.0603, -0.3750,  ..., -0.1592,  0.7227, -0.1289],
       dtype=torch.float16)
[2048], tensor([-0.7344, -0.6172, -1.3984,  ..., -0.1348,  0.7148,  0.3984],
       dtype=torch.float16)
[2048], tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16)
[2048], tensor([-0.4121, -0.1738,  0.4512,  ..., -0.0234, -0.3164,  0.1992],
       dtype=torch.float16)
[2048], tensor([ 0.1611,

# Cascade

In [6]:
from reliable_monitoring.dataset import load_dataset, sample_from_dataset

train_dataset = sample_from_dataset(
    load_dataset(
        Path(f"{DATA_DIR}/training/prompts_4x/train.jsonl"),
        activation_config=activation_config,
    ),
    100,
)

# calibration and test are both exchangeable (actually i.i.d.)
calibration_dataset = load_dataset(
    # Path(f"{DATA_DIR}/evals/dev/toolace_balanced_apr_22.jsonl"),
    Path(f"{DATA_DIR}/evals/dev/anthropic_balanced_apr_23.jsonl"),
    activation_config=activation_config,
)
calibration_dataset = sample_from_dataset(
    calibration_dataset,
    100,
)

test_dataset = load_dataset(
    # Path(f"{DATA_DIR}/evals/test/toolace_test_balanced_apr_22.jsonl"),
    Path(f"{DATA_DIR}/evals/test/anthropic_test_balanced_apr_23.jsonl"),
    activation_config=activation_config,
)
test_dataset = sample_from_dataset(
    test_dataset,
    100,
)

FileNotFoundError: Activations not found for meta-llama/Llama-3.2-1B-Instruct layer 11 on /storage/nvme/prj/llm_monitoring/models-under-pressure/data/evals/dev/anthropic_balanced_apr_23.jsonl. Set auto_compute=True to compute them automatically.

In [7]:
from reliable_monitoring.dataset import split_dataset

opt_dataset, calibration_dataset = split_dataset(calibration_dataset, [0.5, 0.5])

In [8]:
from reliable_monitoring.probes import SequenceProbe

probe = SequenceProbe(reduction_strategy="mean")
probe.fit(train_dataset)

Reducing activations: 100%|██████████| 1/1 [00:00<00:00,  3.89it/s]


In [ ]:
# run cascade evaluation

from reliable_monitoring.cascade import run_llm_baseline, run_offline_cascade
from reliable_monitoring.risks import AccuracyRisk, BudgetCostRisk, RiskEvaluationContext

# Compute probe scores and baseline scores separately
print("Computing probe scores...")
probe_scores = probe.predict(opt_dataset)

print("Computing baseline scores...")
baseline_scores = run_llm_baseline(
    baseline_model_name=MODEL_NAME,
    dataset=opt_dataset,
    baseline_batch_size=16,
)

In [ ]:
thresholds = np.linspace(0.5, 1, 11)
budget_scores = []
empirical_performance_scores = []

for i, t in enumerate(thresholds):
    print(f"Evaluating threshold: {t:.2f}, {i} of {len(thresholds)}")
    opt_scores = run_offline_cascade(
        probe_scores,
        baseline_scores,
        threshold=t,
        merge_strategy="avg",
    )
    # Use empirical computation functions directly
    context = RiskEvaluationContext(cascade_scores=opt_scores, dataset=opt_dataset)
    budget_scores.append(BudgetCostRisk.empirical_computation(context))
    empirical_performance_scores.append(1.0 - AccuracyRisk.empirical_computation(context))

In [12]:
from reliable_monitoring.learn_then_test import is_pareto

costs = np.concatenate(
    [
        -np.array(budget_scores).reshape(-1, 1),  # we want to minimise the budget cost
        np.array(empirical_performance_scores).reshape(-1, 1),
    ],
    axis=1,
)


pareto_front = is_pareto(-costs)
pareto_front

array([ True,  True,  True, False,  True, False,  True,  True, False,
       False, False])

In [13]:
costs

array([[-0.  ,  0.54],
       [-0.06,  0.56],
       [-0.1 ,  0.58],
       [-0.16,  0.58],
       [-0.26,  0.6 ],
       [-0.38,  0.6 ],
       [-0.44,  0.62],
       [-0.52,  0.64],
       [-0.58,  0.64],
       [-0.74,  0.64],
       [-1.  ,  0.64]])

In [14]:
import plotly.express as px

fig = px.scatter(
    x=-costs[:, 0],  # Budget cost (negative because we minimized -cost)
    y=costs[:, 1],  # Empirical performance
    title="Budget Cost vs Empirical Performance",
    labels={"x": "Budget Cost", "y": "Empirical Performance"},
)
fig.add_scatter(
    x=-costs[pareto_front, 0],
    y=costs[pareto_front, 1],
    mode="markers",
    marker=dict(color="red", size=10, symbol="star"),
    name="Pareto Front",
)
fig.show()

In [15]:
budget_scores

[0.0, 0.06, 0.1, 0.16, 0.26, 0.38, 0.44, 0.52, 0.58, 0.74, 1.0]

In [16]:
empirical_performance_scores

[0.54, 0.56, 0.58, 0.58, 0.6, 0.6, 0.62, 0.64, 0.64, 0.64, 0.64]

In [17]:
# now define the empirical risk functions that we will use to find the pareto front and run LTT

# LTT

In [8]:
from pathlib import Path

from reliable_monitoring.dataset import load_dataset, sample_from_dataset

train_dataset = sample_from_dataset(
    load_dataset(
        Path(f"{DATA_DIR}/training/prompts_4x/train.jsonl"),
        activation_config=activation_config,
    ),
    100,
)

# calibration and test are both exchangeable (actually i.i.d.)
calibration_dataset = load_dataset(
    # Path(f"{DATA_DIR}/evals/dev/toolace_balanced_apr_22.jsonl"),
    Path(f"{DATA_DIR}/evals/dev/anthropic_balanced_apr_23.jsonl"),
    activation_config=activation_config,
)
calibration_dataset = sample_from_dataset(
    calibration_dataset,
    100,
)

test_dataset = load_dataset(
    # Path(f"{DATA_DIR}/evals/test/toolace_test_balanced_apr_22.jsonl"),
    Path(f"{DATA_DIR}/evals/test/anthropic_test_balanced_apr_23.jsonl"),
    activation_config=activation_config,
)
test_dataset = sample_from_dataset(
    test_dataset,
    100,
)

In [ ]:
from reliable_monitoring.risks import AccuracyRisk, BudgetCostRisk

controlled_risk = BudgetCostRisk
# for now we don't have an optimized risk

params = np.linspace(0, 1, 11)[1:-1]  # Exclude 0 and 1
# calculate the empirical risks
# calculate the p-values for each param setting using the HB bound on the calibration set
# run FST to find the reliable parameters